# 📓 Notebook 05 — Advanced RAG---**Phase 2 · Retrieval-Augmented Generation**> Basic RAG works but has failure modes. Advanced RAG fixes them with hybrid search, reranking, and context optimization.### What You'll Learn| Topic | Why It Matters ||-------|---------------|| Hybrid search | Combine keyword (BM25) + semantic search || Metadata filtering | Filter by source, date, category before ranking || Reranking | Cross-encoder reranking for precision || Context optimization | Fit more relevant info in the context window || Query transformation | Improve retrieval through query expansion |### 🏗️ Build: Production-Style RAG Pipeline

## 1. Setup

In [ ]:
import os, json, re, timeimport numpy as npfrom dotenv import load_dotenvimport litellmfrom collections import Counterload_dotenv()DEFAULT_MODEL = os.getenv("DEFAULT_MODEL", "gpt-4o-mini")DEFAULT_EMBEDDING = os.getenv("DEFAULT_EMBEDDING_MODEL", "text-embedding-3-small")def get_embedding(text, model=None):    model = model or DEFAULT_EMBEDDING    return litellm.embedding(model=model, input=[text]).data[0]["embedding"]def get_embeddings_batch(texts, model=None):    model = model or DEFAULT_EMBEDDING    return [d["embedding"] for d in litellm.embedding(model=model, input=texts).data]def cosine_similarity(a, b):    a, b = np.array(a), np.array(b)    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))print(f"🤖 LLM: {DEFAULT_MODEL} | 📐 Embeddings: {DEFAULT_EMBEDDING}")

---## 2. The Problem with Basic RAGBasic RAG (semantic search only) fails when:| Failure Mode | Example | Why ||-------------|---------|-----|| **Exact match needed** | "Error code 0x8007" | Semantic search finds *similar meaning*, not exact strings || **Keyword-heavy queries** | "Python pandas groupby aggregate" | Technical terms need keyword matching || **Too many results** | Top-5 chunks are all slightly relevant | Need reranking to find the *best* match || **Context overflow** | 10 relevant chunks but only 4K tokens left | Need compression/selection |> **Interview Tip:** *"What are the failure modes of semantic search?"*  > 1) Fails on exact matches (error codes, proper nouns) 2) Poor with technical jargon 3) Can't filter by metadata 4) No relevance gradation among top results

---## 3. Hybrid Search: BM25 + Semantic### BM25 — The Keyword Search AlgorithmBM25 (Best Matching 25) is a ranking function for keyword search. It's like TF-IDF but better:- **Term Frequency**: Words appearing more in a document score higher- **Inverse Document Frequency**: Rare words score higher than common ones- **Document Length Normalization**: Doesn't favor longer documents

In [ ]:
# BM25 Implementationimport mathclass BM25:    """BM25 keyword search implementation."""    def __init__(self, k1=1.5, b=0.75):        self.k1 = k1        self.b = b        self.docs = []        self.doc_lengths = []        self.avg_dl = 0        self.df = {}   # document frequency        self.idf = {}        self.doc_term_freqs = []        def _tokenize(self, text):        return re.findall(r'\w+', text.lower())        def fit(self, documents):        """Index documents for BM25 search."""        self.docs = documents        n = len(documents)                for doc in documents:            tokens = self._tokenize(doc)            self.doc_lengths.append(len(tokens))                        freq = Counter(tokens)            self.doc_term_freqs.append(freq)                        for token in set(tokens):                self.df[token] = self.df.get(token, 0) + 1                self.avg_dl = sum(self.doc_lengths) / n if n > 0 else 0                for term, df in self.df.items():            self.idf[term] = math.log((n - df + 0.5) / (df + 0.5) + 1)                print(f"  📑 BM25 indexed {n} documents, {len(self.df)} unique terms")        def score(self, query):        """Score all documents against a query."""        query_tokens = self._tokenize(query)        scores = []                for i, doc_tf in enumerate(self.doc_term_freqs):            score = 0            dl = self.doc_lengths[i]                        for token in query_tokens:                if token not in doc_tf:                    continue                tf = doc_tf[token]                idf = self.idf.get(token, 0)                numerator = tf * (self.k1 + 1)                denominator = tf + self.k1 * (1 - self.b + self.b * dl / self.avg_dl)                score += idf * numerator / denominator                        scores.append((i, score))                scores.sort(key=lambda x: x[1], reverse=True)        return scoresprint("✅ BM25 implementation ready")

In [ ]:
class HybridRAG:    """Production RAG with hybrid search (BM25 + Semantic)."""        def __init__(self, model=None, embedding_model=None, semantic_weight=0.7):        self.llm_model = model or DEFAULT_MODEL        self.emb_model = embedding_model or DEFAULT_EMBEDDING        self.semantic_weight = semantic_weight        self.keyword_weight = 1 - semantic_weight        self.chunks = []        self.embeddings = []        self.metadata = []        self.bm25 = BM25()        def ingest(self, documents, metadata_list=None):        """Ingest documents with both semantic and keyword indexing."""        self.chunks = documents        self.metadata = metadata_list or [{} for _ in documents]                # Semantic index        self.embeddings = get_embeddings_batch(documents, model=self.emb_model)                # BM25 index        self.bm25.fit(documents)                print(f"  ✅ Hybrid index: {len(documents)} chunks (semantic + BM25)")        def _normalize_scores(self, scores):        """Min-max normalize scores to [0, 1]."""        if not scores:            return scores        values = [s for _, s in scores]        min_v, max_v = min(values), max(values)        if max_v == min_v:            return [(idx, 1.0) for idx, _ in scores]        return [(idx, (s - min_v) / (max_v - min_v)) for idx, s in scores]        def retrieve(self, query, top_k=5, metadata_filter=None):        """Hybrid retrieval with optional metadata filtering."""        # Semantic scores        query_emb = get_embedding(query, model=self.emb_model)        sem_scores = [(i, cosine_similarity(query_emb, e)) for i, e in enumerate(self.embeddings)]        sem_norm = self._normalize_scores(sem_scores)                # BM25 scores        bm25_scores = self.bm25.score(query)        bm25_norm = self._normalize_scores(bm25_scores)                # Combine scores        combined = {}        for idx, score in sem_norm:            combined[idx] = combined.get(idx, 0) + score * self.semantic_weight        for idx, score in bm25_norm:            combined[idx] = combined.get(idx, 0) + score * self.keyword_weight                # Apply metadata filter        if metadata_filter:            combined = {idx: score for idx, score in combined.items()                        if all(self.metadata[idx].get(k) == v for k, v in metadata_filter.items())}                # Sort and return        ranked = sorted(combined.items(), key=lambda x: x[1], reverse=True)[:top_k]                return [{            "chunk_id": idx,            "text": self.chunks[idx],            "score": round(score, 4),            "metadata": self.metadata[idx],        } for idx, score in ranked]        def query(self, question, top_k=3, metadata_filter=None, verbose=True):        """Full hybrid RAG pipeline."""        chunks = self.retrieve(question, top_k=top_k, metadata_filter=metadata_filter)                if verbose:            print(f"  🔍 Retrieved {len(chunks)} chunks:")            for c in chunks:                print(f"     [{c['score']:.3f}] {c['text'][:60]}...")                context = "\n\n---\n\n".join([c["text"] for c in chunks])                response = litellm.completion(            model=self.llm_model,            messages=[                {"role": "system", "content": "Answer questions using ONLY the provided context. Cite the relevant part. If uncertain, say so."},                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}\nAnswer:"}            ],            temperature=0, max_tokens=500,        )                answer = response.choices[0].message.content        if verbose:            print(f"  📝 {answer}")                return {"answer": answer, "chunks": chunks}print("✅ HybridRAG ready")

In [ ]:
# Test: Hybrid search advantage over pure semanticknowledge = [    "Python error code 0x80070005 means ACCESS_DENIED. Run as administrator to fix.",    "Machine learning models can overfit when training data is too small.",    "The pandas groupby function aggregates data by specified columns.",    "CUDA out of memory error occurs when GPU RAM is exhausted. Reduce batch size.",    "React useState hook manages component-level state in functional components.",    "Docker error 'port already in use' — kill the process using that port or change the port mapping.",    "SQL JOIN combines rows from two tables based on a related column.",    "Git merge conflicts happen when the same lines are edited in different branches.",    "REST API status code 429 means Too Many Requests — implement rate limiting or backoff.",    "Kubernetes pod CrashLoopBackOff indicates the container keeps failing on startup.",]metadata = [    {"category": "windows", "type": "error"},    {"category": "ml", "type": "concept"},    {"category": "python", "type": "howto"},    {"category": "ml", "type": "error"},    {"category": "javascript", "type": "concept"},    {"category": "devops", "type": "error"},    {"category": "database", "type": "concept"},    {"category": "git", "type": "howto"},    {"category": "api", "type": "error"},    {"category": "devops", "type": "error"},]hybrid = HybridRAG(semantic_weight=0.6)hybrid.ingest(knowledge, metadata)# Test queries that need keyword matchingprint("\n" + "=" * 70)print("🔎 Hybrid vs Semantic-Only Comparison")test_queries = [    "error 0x80070005",           # Exact match needed — BM25 shines    "how to aggregate data in pandas",  # Keywords + semantics    "container keeps crashing",    # Semantic understanding needed]for q in test_queries:    print(f"\n❓ {q}")    hybrid.query(q, top_k=2)    print("-" * 50)

In [ ]:
# Metadata filtering demoprint("🏷️ Metadata Filtering Demo")print("=" * 50)# Only search error-type documentsprint("\n📋 Filter: type=error only")hybrid.query("something is failing", top_k=3, metadata_filter={"type": "error"})print("\n📋 Filter: category=devops only")hybrid.query("deployment issue", top_k=2, metadata_filter={"category": "devops"})

---## 4. Query TransformationSometimes the user's query isn't optimal for retrieval. Transform it!

In [ ]:
def expand_query(query, model=None):    """Use LLM to generate better search queries."""    model = model or DEFAULT_MODEL    response = litellm.completion(        model=model,        messages=[{            "role": "system",            "content": "Generate 3 alternative search queries that would help find relevant information. Return as a JSON array of strings."        }, {            "role": "user",            "content": f"Original query: {query}"        }],        response_format={"type": "json_object"},        temperature=0.3,    )        try:        result = json.loads(response.choices[0].message.content)        queries = result.get("queries", result.get("alternatives", list(result.values())[0]))        return [query] + queries  # Original + expanded    except:        return [query]# Demo query expansionoriginal = "my app is slow"expanded = expand_query(original)print(f"🔄 Query Expansion")print(f"   Original: {original}")print(f"   Expanded:")for i, q in enumerate(expanded):    print(f"     {i}: {q}")

---## 5. Context Window OptimizationWhen you have many relevant chunks but limited context window.

In [ ]:
def optimize_context(chunks, max_tokens=2000, model=None):    """Fit as many relevant chunks as possible within token budget."""    import tiktoken    try:        encoder = tiktoken.encoding_for_model("gpt-4o")    except:        encoder = tiktoken.get_encoding("cl100k_base")        selected = []    total_tokens = 0        for chunk in chunks:  # Already sorted by relevance        chunk_tokens = len(encoder.encode(chunk["text"]))        if total_tokens + chunk_tokens <= max_tokens:            selected.append(chunk)            total_tokens += chunk_tokens        else:            break        print(f"  📦 Context: {len(selected)}/{len(chunks)} chunks, {total_tokens} tokens")    return selected# Demosample_chunks = [    {"text": "A" * 500, "score": 0.95},    {"text": "B" * 300, "score": 0.88},    {"text": "C" * 400, "score": 0.82},    {"text": "D" * 800, "score": 0.75},    {"text": "E" * 200, "score": 0.70},]optimize_context(sample_chunks, max_tokens=500)

---## 📝 Interview Quiz — Advanced RAG### Q1: What is hybrid search and why is it better than semantic-only?<details><summary>💡 Answer</summary>Hybrid search combines **keyword search (BM25)** and **semantic search (embeddings)**:- BM25 excels at exact matches: error codes, proper nouns, technical terms- Semantic search excels at meaning: paraphrases, conceptual queries- Combined with weighted scoring (e.g., 0.7 semantic + 0.3 keyword)**Why better:** Covers both failure modes. "Error 0x80070005" needs keywords. "My app keeps crashing" needs semantics.</details>### Q2: What is reranking and how does it differ from initial retrieval?<details><summary>💡 Answer</summary>**Initial retrieval** (bi-encoder): Fast but approximate. Embeds query and documents independently.**Reranking** (cross-encoder): Slow but precise. Processes query AND document together in one pass.Pipeline: Fast retrieval gets top-50 → Cross-encoder reranks to top-5.Cross-encoders are 100x slower but significantly more accurate because they see both texts simultaneously, enabling fine-grained attention between query and document tokens.</details>### Q3: How would you handle a RAG system with millions of documents?<details><summary>💡 Answer</summary>1. **Vector DB** (FAISS, Pinecone) for approximate nearest neighbor (ANN) search2. **Pre-filtering** with metadata (date, category, source) before vector search3. **Hierarchical indexing**: Document-level summary embeddings → Chunk-level search within matched docs4. **Caching**: Cache frequent query embeddings and results5. **Sharding**: Distribute index across multiple machines6. **Incremental updates**: Only re-embed changed documents</details>### Q4: What is query transformation? Give 3 techniques.<details><summary>💡 Answer</summary>1. **Query expansion**: Use LLM to generate alternative search queries, search all, merge results2. **HyDE** (Hypothetical Document Embedding): Generate a hypothetical answer, embed THAT, search with it3. **Step-back prompting**: Ask a more general question to find broader context4. **Query decomposition**: Split complex queries into sub-queries, search each independently**Example:** "Why is Python slow?" → Also searches: "Python performance bottlenecks", "Python interpreter speed", "CPython GIL limitations"</details>

---## ✅ Notebook 05 Summary| Concept | Key Takeaway ||---------|-------------|| Hybrid search | BM25 + semantic covers both keyword and meaning queries || BM25 | Classic keyword ranking with TF-IDF + length normalization || Metadata filtering | Pre-filter by source/date/category before ranking || Reranking | Cross-encoder second pass for precision (top-50 → top-5) || Query transformation | Expand/rewrite queries for better retrieval || Context optimization | Fit max relevant info within token budget |### ➡️ Next: [Notebook 06 — Vector Databases](./06_vector_db.ipynb)